<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/module_75_a2a_protocol_agent_communication.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🛰️ **Module 75 — A2A Protocol + Agent Communication** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)

> Phase 10 · Master-Plan Gaps


# Module 75 — A2A + Agent Communication

> **MCP (M64)** standardised how a model talks to **tools**. **A2A (Agent-to-Agent)** — Google's open protocol announced in April 2025 — standardises how one **agent** talks to **another agent**. Imagine an HR agent at Company A asking a payroll agent at Company B to file a tax form, then handing the result to a legal-review agent at Company C — across organisations, with auth, capability discovery, and streaming progress. That's A2A's pitch.
>
> By late 2025 the protocol was donated to the **Linux Foundation** with backing from Microsoft, Salesforce, SAP, Atlassian, Cisco, MongoDB and ~50 others — making it the de-facto standard. This module is the practical map.

### What you'll cover
1. **A2A vs MCP** — different problem, different layer
2. The four primitives — **Agent Card**, **Task**, **Message**, **Artifact**
3. The protocol surface — **JSON-RPC over HTTP** + **SSE**
4. **Agent Cards** — `/.well-known/agent.json` as the manifest
5. Task lifecycle — **submitted → working → input-required → completed / failed**
6. **Messages** — text · file · data parts; multi-turn conversation
7. **Streaming** — SSE for long-running work
8. **Auth + identity + trust** — bearer / mTLS / signed cards / Agent Passport
9. Building a small A2A server in Python
10. A2A vs MCP vs LangGraph hand-off — when to use what
11. The 2025 ecosystem + future direction (AGNTCY, ACP, Cisco AgentFlow)


## 1 · A2A vs MCP — different problem, different layer

| | **MCP (M64)** | **A2A (this module)** |
|---|---|---|
| Who talks to whom | **Model ↔ Tool** | **Agent ↔ Agent** |
| Owned by | the host (Claude / Cursor / VS Code) | the agent itself (often another org) |
| Primitives | tools · resources · prompts | tasks · messages · artifacts |
| Trust boundary | inside one host process | **across organisations** |
| Discovery | enumerated by host config | **`/.well-known/agent.json`** (Agent Card) |
| Auth | host-level | per-agent OAuth / bearer / mTLS |
| Streaming | tool result | **task progress + intermediate artifacts** |
| Multi-turn? | one tool call | full conversation with `input-required` states |

**The relationship.** MCP is the *plumbing inside one agent* (model → tools, files, resources). A2A is the *wire between agents*. A production multi-agent system uses **both**: each agent uses MCP internally to access its own tools, and exposes an A2A endpoint so other agents can call it.

```
   Agent A ◄── MCP ──► its tools (Slack, DB, GitHub)
       │
       │ A2A (this module)
       ▼
   Agent B ◄── MCP ──► its tools (Salesforce, Jira)
```


## 2 · The four primitives

| Primitive | What it is |
|---|---|
| **Agent Card** | JSON manifest at `/.well-known/agent.json` — name, description, skills, auth, endpoints |
| **Task** | a unit of work; has a UUID `taskId` and a state machine |
| **Message** | a turn in the conversation — list of typed `parts` (text / file / data) |
| **Artifact** | a typed output produced by a task — files, JSON blobs, references |

You'll think in terms of: *Agent A discovers Agent B's card → submits a Task with an initial Message → polls or streams the Task's state → receives Artifacts when done.*

```
   discover  →  submit task  →  stream / poll updates  →  receive artifact
       ▼              ▼                    ▼                       ▼
  /well-known     POST /tasks          SSE / poll              .result
```


## 3 · The protocol surface

A2A uses **JSON-RPC 2.0 over HTTPS** (the same family as MCP, deliberately — Google wanted them to feel familiar). For streaming results, it uses **Server-Sent Events (SSE)** on a sub-path of the same endpoint.

Core methods:

| Method | Purpose |
|---|---|
| `tasks/send` | submit a new task; return the initial state |
| `tasks/sendSubscribe` | submit a task **and** stream updates over SSE |
| `tasks/get` | poll a task's current state |
| `tasks/cancel` | request cancellation |
| `tasks/resubscribe` | reconnect to an in-flight task's SSE stream after a network blip |
| `tasks/pushNotificationConfig/set` | configure webhooks for state changes |

All methods are POSTs to the same endpoint (e.g. `https://agent-b.example.com/a2a`). The Agent Card tells you that URL.

## 4 · Agent Cards — discovery

In [ ]:
# /.well-known/agent.json — what Agent B publishes about itself
agent_card = {
    "name": "Payroll Agent",
    "description": "Files tax forms, computes withholdings, exports W2s.",
    "url": "https://agent-b.example.com/a2a",
    "version": "1.4.0",
    "provider": {
        "organization": "Acme Payroll",
        "url": "https://acme.example.com",
    },
    "capabilities": {
        "streaming": True,
        "pushNotifications": True,
        "stateTransitionHistory": True,
    },
    "authentication": {
        "schemes": ["bearer", "mTLS"],
    },
    "defaultInputModes":  ["text/plain", "application/json"],
    "defaultOutputModes": ["text/plain", "application/json", "application/pdf"],
    "skills": [
        {
            "id": "file_w2",
            "name": "File W-2 form",
            "description": "Submits a W-2 to the IRS for a given employee record.",
            "inputModes":  ["application/json"],
            "outputModes": ["application/pdf"],
            "examples": ["Please file a W-2 for employee #123 for tax year 2025."],
        },
    ],
}
print(json.dumps(agent_card, indent=2)[:500] + " …")

**Why a manifest.** An agent looking to delegate work fetches `https://agent-b.example.com/.well-known/agent.json` once, caches it, and uses it to:
- pick the right skill (`skills[].id`)
- format the input correctly (`defaultInputModes`)
- negotiate auth (`authentication.schemes`)
- know whether it can stream (`capabilities.streaming`)

`/.well-known/` is the standard place to put metadata for **any HTTPS service** (the [RFC 8615](https://datatracker.ietf.org/doc/html/rfc8615) convention also used by OAuth, OIDC, ACME, password-managers). A2A picked it on purpose.

## 5 · Task lifecycle

```
        ┌──────────┐    ┌──────────┐    ┌──────────────────┐    ┌────────────┐
   →    │submitted │ →  │ working  │ →  │ input-required   │ →  │ completed  │
        └──────────┘    └─────┬────┘    └────────┬─────────┘    └────────────┘
                              │                  │
                              ▼                  ▼
                         ┌────────┐         ┌────────┐
                         │ failed │         │canceled│
                         └────────┘         └────────┘
```

| State | What it means |
|---|---|
| `submitted` | task accepted; not yet running |
| `working` | agent is executing |
| `input-required` | agent paused for more info — human or another agent must respond |
| `completed` | finished with artifacts |
| `failed` | something went wrong (`error` populated) |
| `canceled` | requester cancelled |

**The `input-required` state is the killer feature.** A long-running task can pause, ask "do you want me to file the W-2 in the employee's name or the dependent's?", and resume when the answer arrives — minutes, hours, or days later. Without this, multi-agent workflows degenerate into hand-rolled polling logic.

## 6 · Messages — multimodal parts

In [ ]:
# A Message is a list of typed parts — text / file / structured data
example_message = {
    "role": "user",
    "parts": [
        { "type": "text",
          "text": "Please file a W-2 for the attached employee record." },
        { "type": "file",
          "file": {
            "name": "employee_123.json",
            "mimeType": "application/json",
            "bytes": "eyJlbXBsb3llZUlkIjogMTIzfQ=="   # base64
          }
        },
        { "type": "data",
          "data": { "taxYear": 2025, "preferredFilingMethod": "e-file" }
        },
    ],
}
print(json.dumps(example_message, indent=2)[:300] + " …")

**Three part types** cover almost everything an agent needs to send:

- **`text`** — natural-language instructions or replies.
- **`file`** — base64 payload **or** a `uri` reference to S3 / signed URL. Use `uri` for big files.
- **`data`** — typed structured data (JSON Schema validatable). Use this for tool arguments, function results, or anything machine-readable.

The same shape is used for **outgoing** messages (your turn) and **incoming** messages (the other agent's reply). Artifacts on completion are also lists of parts.

## 7 · Streaming with SSE

A long task that takes ten minutes shouldn't block. A2A's answer: `tasks/sendSubscribe` opens an **SSE** stream that pushes events as the task progresses.

```http
POST /a2a HTTP/1.1
Content-Type: application/json
Accept: text/event-stream

{ "jsonrpc": "2.0", "id": 1, "method": "tasks/sendSubscribe",
  "params": { "id": "task-xyz", "message": { ... } } }
```

The server responds with an SSE stream of `data:` lines:

```
event: TaskStatusUpdate
data: { "id": "task-xyz", "status": { "state": "working", "message": ... } }

event: TaskArtifactUpdate
data: { "id": "task-xyz", "artifact": { "parts": [ { "type":"text", "text":"Form drafted…" } ] } }

event: TaskStatusUpdate
data: { "id": "task-xyz", "status": { "state": "completed", "finalArtifacts": [...] } }
```

Three event types:
- **`TaskStatusUpdate`** — state changed (working → input-required, etc.).
- **`TaskArtifactUpdate`** — a new partial artifact is available.
- **`TaskMessage`** — the agent sent a message back (questions during `input-required`).

If the client drops, **`tasks/resubscribe`** reconnects on the same `taskId` and replays missed events.

## 8 · Auth + identity + trust

A2A inherits HTTPS's auth ecosystem. The Agent Card declares what schemes are supported in `authentication.schemes`.

| Scheme | When |
|---|---|
| **Bearer token** | simple SaaS-to-SaaS, short-lived OAuth access tokens |
| **mTLS** | cross-org, regulated industries; certs proven by the connecting client |
| **OAuth 2.0** flows | when delegation chains need user-on-behalf-of semantics |
| **Signed Agent Cards** | the agent's identity is signed by a trusted CA — the requester can verify the card hasn't been tampered with |
| **Agent Passport** (proposed extension) | scoped, attenuated tokens that pass through long delegation chains |

Three rules that survive contact with production:

1. **Never trust the agent's self-declared identity** — the card is metadata; the auth scheme is the truth.
2. **Use scoped tokens per skill** — a token valid for `file_w2` shouldn't work for `delete_employee`. Same principle as M74 tool design.
3. **Audit every cross-agent call** — log `(callerAgentId, calleeAgentId, skillId, taskId, traceId)` for incident response. Pair with OTel (M51).

The protocol explicitly does **not** prescribe an identity provider. You bring your own (Auth0, Okta, Entra, Keycloak, AWS Cognito, GCP IAM, …) and put it behind a token-verifying middleware.

## 9 · A tiny A2A server + client in Python

The official SDK is `a2a-sdk` (also published as `google-a2a` on PyPI in early 2025; renamed during Linux Foundation transfer). The shape:

In [ ]:
# pip install a2a-sdk fastapi uvicorn

# SERVER side — wrap a function as a skill, expose at /a2a
server_sketch = '''
from a2a_sdk.server import A2AServer, Skill, TaskContext
from fastapi import FastAPI

server = A2AServer(
    name="Math Agent",
    description="Adds, multiplies, factors numbers.",
    url="https://math.example.com/a2a",
)

@server.skill(
    id="add",
    name="Add two numbers",
    description="Returns a + b.",
    input_schema={"type":"object","required":["a","b"],
                  "properties":{"a":{"type":"number"},"b":{"type":"number"}}},
)
async def add(ctx: TaskContext):
    msg = ctx.message
    payload = next(p for p in msg.parts if p.type == "data").data
    a, b = payload["a"], payload["b"]
    await ctx.emit_status("working", "Computing...")
    result = a + b
    await ctx.emit_artifact({"type":"data","data":{"sum": result}})
    await ctx.emit_status("completed")

app = FastAPI()
server.mount(app, path="/a2a")
# uvicorn server:app --port 9000
'''
print(server_sketch)

In [ ]:
# CLIENT side — fetch the card, call the skill, stream the result
client_sketch = '''
from a2a_sdk.client import A2AClient

async def main():
    # 1) Discover
    client = await A2AClient.from_well_known("https://math.example.com")
    print("agent:", client.card.name)
    print("skills:", [s.id for s in client.card.skills])

    # 2) Submit + stream
    async for evt in client.send_task_subscribe(
        skill_id="add",
        message={"parts":[{"type":"data","data":{"a": 2, "b": 3}}]},
    ):
        if evt.kind == "status":
            print("status →", evt.status.state)
        elif evt.kind == "artifact":
            print("artifact →", evt.artifact)
        elif evt.kind == "final":
            print("done   →", evt.final_artifacts)
            break
'''
print(client_sketch)

**Note the shape of the skill handler.** It receives a **`TaskContext`** — not just inputs and outputs. The context lets the skill:
- read the incoming message (multi-part)
- emit status updates (`ctx.emit_status(...)`)
- emit interim artifacts
- pause for input (`await ctx.request_input(prompt=...)`)
- access the original requester's identity (`ctx.caller`)

That's the surface area that lets long-running, multi-turn, cross-org agent workflows feel coherent.

## 10 · A2A vs MCP vs LangGraph hand-off — when to use what

| Need | Pick | Why |
|---|---|---|
| Model ↔ tool inside one host | **MCP (M64)** | unified tool standard; host owns trust |
| Agents inside one Python process | **LangGraph / crewAI (M33, M43)** | in-process function calls; no network |
| Agents across processes in one company | **A2A** — *or* **gRPC (M45)** | A2A if you want capability discovery + lifecycle; gRPC if it's just RPC |
| Agents across companies | **A2A** | discovery + auth + lifecycle baked in |
| Marketplace of public agents | **A2A** + signed cards | the only model designed for adversarial trust |
| One-shot REST hit | **plain REST** | A2A is overkill for stateless requests |

### Common confusion
- **A2A is not a replacement for MCP.** A production agent uses both — MCP-in, A2A-out.
- **A2A is not a framework.** Frameworks like LangGraph, crewAI, AutoGen, Pydantic AI, Mastra speak A2A as a *transport*. You can build an A2A-compatible agent on top of any of them.
- **A2A doesn't include scheduling.** If you need durable retries / saga orchestration, run A2A inside an orchestrator (M68): Temporal, Airflow, Inngest, Trigger.dev.

## 11 · The 2025 ecosystem + future direction

The agent-comm space crystallised in 2024-25. Three converging stacks:

| Stack | Backed by | Status |
|---|---|---|
| **A2A** | Google (originator), then Linux Foundation, MS, Salesforce, SAP, Atlassian, Cisco, MongoDB, +50 | **De-facto standard in 2025** |
| **MCP** | Anthropic | tools-side standard, now adopted by OpenAI too |
| **ACP (Agent Communication Protocol)** | IBM (research) | research-stage; smaller community |
| **AGNTCY (Internet of Agents)** | Cisco + LangChain | sits *above* A2A — directory + identity + sk |
| **NLWeb / .well-known/ai-agent.json** | Microsoft / OpenAI proposals | tangential conventions for agent metadata on the open web |

### What to internalise as "the 2026 default"

- **MCP for tools, A2A for agents.** Two protocols, one stack.
- Each of your agents publishes an Agent Card; each of your hosts mounts an A2A endpoint.
- Identity rides on top via **bearer / mTLS / signed cards** — your IdP, not the protocol.
- Cross-organisation? Use the **AGNTCY directory** to discover agents, then call them with A2A.

### Open questions to track (2026+)
- **Pricing primitives** — A2A doesn't define payments. Watch [x402](https://x402.org) and Skyfire as proposals for "pay to call this agent."
- **Capability attenuation** — Agent Passport / Macaroon-style scoped tokens for long delegation chains.
- **Adversarial agent discovery** — what stops a malicious agent from publishing a misleading card? Signed cards + Web of Trust attestations are early answers.

## ✅ Recap
- **A2A** is Google's open agent-to-agent protocol; donated to the Linux Foundation in 2025 with 50+ vendor backers.
- It's **JSON-RPC over HTTPS** + **SSE for streaming** — the same family as MCP.
- Four primitives: **Agent Card**, **Task**, **Message** (with `text` / `file` / `data` parts), **Artifact**.
- Task lifecycle has six states; `input-required` is the killer feature for long multi-turn work.
- **MCP ≠ A2A.** MCP = model↔tool; A2A = agent↔agent. Production agents use **both**.
- Auth via bearer / mTLS / signed cards / OAuth — A2A doesn't prescribe an IdP.
- 2026 default stack: **MCP-in, A2A-out**, each agent publishes a Card, identity rides on existing IdPs.

Next: **M76 — Classical RL Foundations (MDP → PPO)**.
